# Demo - Train a Tumor/Cyst Classifier on the KiTS23 Data
[KiTS23](https://kits-challenge.org/kits23/) was a competition were teams competed to develop systems for segmentation of kidneys, tumors and cysts.
You can download their dataset [here](https://github.com/neheller/kits23). 

But worry not, we already extracted the features and saved them in [data/kits_radiomics.parquet](../data/kits_radiomics.parquet).


## Feature Handling

### 1. Extract Features
This step requires that you downloaded the KiTS23 data and can be time-intensive.
You may skip this and proceed with **2. Inspect Data**.

In [ ]:
! rv extract \
    --data ../data/KITS_short.csv \
    --output ../data/kits_features.parquet \
    --extractor radiomics \
    --label-map KiTS_label_map.json \
    --augment 0

In [ ]:
! rv extract \
    --data ../data/KITS_short.csv \
    --output ../data/kits_features.parquet \
    --extractor embeddings \
    --label-map KiTS_label_map.json \
    --augment 10

### 2. Inspect Data

In [1]:
import pandas as pd

features = pd.read_parquet("../data/kits_radiomics.parquet")

def describe_data(features):
    n_cases = features['case'].nunique()
    n_lesions = len(features[~features['augmented']])
    print(f"Extracted features for {n_lesions} lesions from {n_cases} cases.")

    classes = features['class_id'].unique()
    print(f"Found {len(classes)} classes: {classes}")

    for cl in classes:
        n_cl_lesions = len(features[(features['class_id'] == cl) & (~features['augmented'])])
        print(f"  Class {cl}: {n_cl_lesions} lesions")

    oversampling_factor = (len(features)-n_lesions) / n_lesions 
    print(f"Each lesion was augmented {oversampling_factor:.1f} times on average.")

describe_data(features)

Extracted features for 1399 lesions from 468 cases.
Found 2 classes: [0 1]
  Class 0: 560 lesions
  Class 1: 839 lesions
Each lesion was augmented 0.0 times on average.


### 3. Split Data
Now that we have features we can split them into a train and a test partition. We also remove all augmentations (if any) from the test partition

In [2]:

from renal_vision.shared.utils import generate_stratified_group_split

train, test = generate_stratified_group_split(features,group_col="case")
test = test[~test['augmented']].reset_index(drop=True)

train.to_parquet("features_train.parquet", index=False)
test.to_parquet("features_test.parquet", index=False)

print("Train set:")
describe_data(train)
print("\nTest set:")
describe_data(test)

Train set:
Extracted features for 1154 lesions from 375 cases.
Found 2 classes: [0 1]
  Class 0: 441 lesions
  Class 1: 713 lesions
Each lesion was augmented 0.0 times on average.

Test set:
Extracted features for 245 lesions from 93 cases.
Found 2 classes: [0 1]
  Class 0: 119 lesions
  Class 1: 126 lesions
Each lesion was augmented 0.0 times on average.


## Training

### 1. [Optional] Generate Task Specifications
Lets specify some meta data that helps us to understand the models output better.

In [3]:
import json

names_binary = {
    0 : "Cyst",
    1 : "Tumor",
}

with open("names_binary.json","w") as f:
    json.dump(names_binary, f)    

### 2. Train Models

In [4]:
! rv train \
    --data features_train.parquet \
    --model xgboost \
    --output-dir binary_model \
    --class-config "names_binary.json"

Loading training data from features_train.parquet...
Detected 88 features: ['original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Median', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RobustMeanAbsoluteDeviation', 'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness', 'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity', 'original_firstorder_Variance', 'original_glcm_Autocorrelation', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_ClusterTendency', 'original_glcm_Contrast', 'original_glcm_Correlation', 'original_glcm_DifferenceAverage', 'original_glcm_DifferenceEntropy', 'original_glcm_DifferenceVariance', 'origina

## Evaluate Models

In [5]:
# jupyter IPython may throw an error when plotting the confusion matrix.
# If that happens, please run the following command in a terminal instead.
! rv eval \
    --data features_test.parquet \
    --model binary_model/model.pkl \
    --output-dir binary_model

Loading test data from features_test.parquet...
Loading model from binary_model/model.pkl...
Running predictions...

Accuracy: 0.9061
F1 Score: 0.9052
Classification Report:
              precision    recall  f1-score   support

        Cyst       0.99      0.82      0.89       119
       Tumor       0.85      0.99      0.92       126

    accuracy                           0.91       245
   macro avg       0.92      0.90      0.90       245
weighted avg       0.92      0.91      0.91       245

Traceback (most recent call last):
  File "/home/haha16/.conda/envs/TumorCyst/bin/rv", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/haha16/kidney/CystClassifier/src/renal_vision/cli.py", line 22, in main
    args.func(args)
  File "/home/haha16/kidney/CystClassifier/src/renal_vision/modeling/cli.py", line 25, in run_eval
    eval.run_evaluation(
  File "/home/haha16/kidney/CystClassifier/src/renal_vision/modeling/eval.py", line 100, in run_evaluation
    plot_confu